In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus/10_Travel_and_Expense_Policy.pdf
/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus/03_Work_From_Home_Policy.pdf
/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus/09_Onboarding_and_Separation_Policy.pdf
/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus/07_IT_and_Data_Security_Policy.pdf
/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus/06_Compensation_and_Benefits_Policy.pdf
/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus/sample_submission.xlsx
/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus/04_Code_of_Conduct.pdf
/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus/01_Employee_Handbook.pdf
/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus/05_Performance

In [7]:
!pip install -q langchain langchain-community langchain-groq langchain-huggingface faiss-cpu pypdf sentence-transformers streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 64.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 55.9 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.3/554.3 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 84.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.

In [8]:
import langchain
import faiss
import sentence_transformers

print("All libraries imported successfully ✅")

All libraries imported successfully ✅


In [9]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

groq_api_key = user_secrets.get_secret("GROQ_API_KEY")

print("Groq key loaded successfully ✅")

Groq key loaded successfully ✅


In [10]:
from langchain_community.document_loaders import PyPDFLoader
import os

pdf_folder = "/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus"

documents = []

for file in os.listdir(pdf_folder):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(pdf_folder, file))
        documents.extend(loader.load())

print("Total pages loaded:", len(documents))

/tmp/ipykernel_58/3321291433.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Total pages loaded: 39


In [11]:
!pip install -q langchain-text-splitters

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 162


In [13]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model loaded ✅")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings model loaded ✅


In [14]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector database created ✅")

FAISS vector database created ✅


In [15]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20
    }
)

print("Retriever created ✅")

Retriever created ✅


In [16]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key=groq_api_key,
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

print("Groq LLM connected ✅")

Groq LLM connected ✅


In [17]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":20}
)

print("Retriever Updated ✅")

Retriever Updated ✅


In [18]:
def ask_hr_bot(question):
    docs = vectorstore.similarity_search(question, k=10)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
You are an HR assistant for Zyro Dynamics.

Answer ONLY using the information in the context.

If the answer is not present in the context, reply exactly:
I can only answer HR-related questions based on Zyro Dynamics policy documents.

Context:
{context}

Question:
{question}

Answer:
"""

    response = llm.invoke(prompt)
    return response.content

In [19]:
docs = vectorstore.similarity_search(
    "How many casual leaves are employees entitled to?",
    k=10
)

context = "\n\n".join([doc.page_content for doc in docs])

print("Casual Leave (CL)" in context)

False


In [20]:
docs = vectorstore.similarity_search(
    "Casual Leave (CL) 8 days",
    k=10
)

context = "\n\n".join([doc.page_content for doc in docs])

print("Casual Leave (CL)" in context)

True


In [21]:
print(
    ask_hr_bot(
        "Casual Leave (CL)"
    )
)

8 days, cannot be carried forward to the next financial year, nor can they be encashed. Any balance remaining at the end of the financial year (31 March) will lapse automatically. Employees are not permitted to avail more than 2 consecutive days of Casual Leave.


In [22]:
docs = retriever.invoke("How many casual leaves are employees entitled to?")

print("Retrieved Documents:", len(docs))

for i, doc in enumerate(docs):
    print(f"\n--- Document {i+1} ---")
    print(doc.page_content[:1000])

Retrieved Documents: 20

--- Document 1 ---
contract employees, probationary employees, interns, and consultants on the company payroll. Employees
seconded from third-party vendors are governed by the terms of their respective contracts.
Leave Cycle: Financial Year (01 April to 31 March)

--- Document 2 ---
month, provided the employee has worked for a minimum of 20 days during the previous month.
 For employees joining or separating during the leave cycle, all leave will be calculated on a pro-rata basis.
 Casual Leave and Sick Leave cannot be carried forward to the next financial year, nor can they be
encashed. Any balance remaining at the end of the financial year (31 March) will lapse automatically.

--- Document 3 ---
15% of CTC
L7
Senior Manager
Rs. 60.0L to Rs. 80.0L
18% of CTC
L8
Director
Rs. 80.0L to Rs. 1.2Cr
20% of CTC
L9
Vice President
Rs. 1.2Cr to Rs. 2.0Cr
25% of CTC
L10
C-Suite
Rs. 2.0Cr and above
30% or more of CTC
STATUTORY BONUS
Employees whose gross monthly salary 

In [23]:
docs = retriever.invoke("casual leave")

print("Retrieved:", len(docs), "documents")

for i, doc in enumerate(docs):
    print(f"\n--- Document {i+1} ---")
    print(doc.page_content[:1000])

Retrieved: 20 documents

--- Document 1 ---
month, provided the employee has worked for a minimum of 20 days during the previous month.
 For employees joining or separating during the leave cycle, all leave will be calculated on a pro-rata basis.
 Casual Leave and Sick Leave cannot be carried forward to the next financial year, nor can they be
encashed. Any balance remaining at the end of the financial year (31 March) will lapse automatically.

--- Document 2 ---
 Employees are not permitted to avail more than 2 consecutive days of Casual Leave. If additional days are
required, they must be clubbed with Earned Leave.
 Sick Leave taken for more than 2 consecutive days requires a Medical Certificate from a registered medical
practitioner, to be submitted within 3 working days of returning to work.
 Leave cannot be availed during the notice period unless specifically approved by the reporting manager and
HR.
EARNED LEAVE (EL)
Eligibility and Accrual

--- Document 3 ---
Not applicable

In [24]:
for i, chunk in enumerate(chunks):
    text = chunk.page_content

    if "Bereavement Leave" in text:
        print("="*60)
        print(text)
        print("="*60)

Zyro Dynamics Pvt. Ltd.
Leave Policy
Doc Code: ZDL-HR-002
Confidential — For Internal Use Only
Page 2
ANNUAL LEAVE ENTITLEMENT
 Leave Type
 Annual Entitlement
 Carry Forward
 Encashable
Earned Leave (EL)
15 days (after 1 year); 1.25
days/month
Up to 45 days
Yes
Casual Leave (CL)
8 days
Not allowed
Not applicable
Sick Leave (SL)
10 days
Not allowed
Not applicable
Maternity Leave
26 weeks
Not applicable
Not applicable
Paternity Leave
10 days
Not applicable
Not applicable
Bereavement Leave
5 days
Not applicable
Bereavement Leave
5 days
Not applicable
Not applicable
Marriage Leave
3 days (once in service)
Not applicable
Not applicable
Compensatory Off
As earned
Up to 7 days (60-day
window)
Not applicable
Loss of Pay (LOP)
As needed
Not applicable
Not applicable
LEAVE RULES AND GUIDELINES
The following general rules apply to all leave types unless specifically stated otherwise:
 Casual Leave and Sick Leave are credited to the employee's leave balance on the first day of the following
withi

In [25]:
import os

for f in os.listdir("/kaggle/working"):
    print(f)

.virtual_documents


In [26]:
print(type(retriever))

<class 'langchain_core.vectorstores.base.VectorStoreRetriever'>


In [27]:
print(type(vectorstore))

<class 'langchain_community.vectorstores.faiss.FAISS'>


In [28]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7cca99e69dc0>, search_kwargs={'k': 20})

In [29]:
docs = retriever.invoke("How many casual leaves are employees entitled to?")

print("Retrieved:", len(docs), "documents\n")

for i, doc in enumerate(docs):
    print(f"\n--- Document {i+1} ---")
    print(doc.page_content[:1000])

Retrieved: 20 documents


--- Document 1 ---
contract employees, probationary employees, interns, and consultants on the company payroll. Employees
seconded from third-party vendors are governed by the terms of their respective contracts.
Leave Cycle: Financial Year (01 April to 31 March)

--- Document 2 ---
month, provided the employee has worked for a minimum of 20 days during the previous month.
 For employees joining or separating during the leave cycle, all leave will be calculated on a pro-rata basis.
 Casual Leave and Sick Leave cannot be carried forward to the next financial year, nor can they be
encashed. Any balance remaining at the end of the financial year (31 March) will lapse automatically.

--- Document 3 ---
15% of CTC
L7
Senior Manager
Rs. 60.0L to Rs. 80.0L
18% of CTC
L8
Director
Rs. 80.0L to Rs. 1.2Cr
20% of CTC
L9
Vice President
Rs. 1.2Cr to Rs. 2.0Cr
25% of CTC
L10
C-Suite
Rs. 2.0Cr and above
30% or more of CTC
STATUTORY BONUS
Employees whose gross monthly salary

In [30]:
docs = retriever.invoke("Casual Leave CL 8 days annual entitlement")

for i, doc in enumerate(docs):
    print(f"\n--- Document {i+1} ---")
    print(doc.page_content[:1500])


--- Document 1 ---
month, provided the employee has worked for a minimum of 20 days during the previous month.
 For employees joining or separating during the leave cycle, all leave will be calculated on a pro-rata basis.
 Casual Leave and Sick Leave cannot be carried forward to the next financial year, nor can they be
encashed. Any balance remaining at the end of the financial year (31 March) will lapse automatically.

--- Document 2 ---
HR.
EARNED LEAVE (EL)
Eligibility and Accrual
Earned Leave is accrued based on the length of continuous service. Employees become eligible for 15 days of
Earned Leave upon completion of one year of continuous service, provided they have worked for a minimum of
240 days in that year. Thereafter, Earned Leave accrues at the rate of 1.25 days per month. Employees in their
probation period accrue EL at 0.5 days per month, which becomes available for use only after probation
confirmation.

--- Document 3 ---
Zyro Dynamics Pvt. Ltd.
Leave Policy
Doc Code

In [31]:
print(ask_hr_bot("How many casual leaves are employees entitled to?"))

I can only answer HR-related questions based on Zyro Dynamics policy documents.


In [32]:
question = "How many casual leaves are employees entitled to?"

docs = retriever.invoke(question)

for i, doc in enumerate(docs):
    print(f"\n===== DOC {i+1} =====")
    print(doc.page_content)


===== DOC 1 =====
contract employees, probationary employees, interns, and consultants on the company payroll. Employees
seconded from third-party vendors are governed by the terms of their respective contracts.
Leave Cycle: Financial Year (01 April to 31 March)

===== DOC 2 =====
month, provided the employee has worked for a minimum of 20 days during the previous month.
 For employees joining or separating during the leave cycle, all leave will be calculated on a pro-rata basis.
 Casual Leave and Sick Leave cannot be carried forward to the next financial year, nor can they be
encashed. Any balance remaining at the end of the financial year (31 March) will lapse automatically.

===== DOC 3 =====
15% of CTC
L7
Senior Manager
Rs. 60.0L to Rs. 80.0L
18% of CTC
L8
Director
Rs. 80.0L to Rs. 1.2Cr
20% of CTC
L9
Vice President
Rs. 1.2Cr to Rs. 2.0Cr
25% of CTC
L10
C-Suite
Rs. 2.0Cr and above
30% or more of CTC
STATUTORY BONUS
Employees whose gross monthly salary is at or below Rs. 21,000 (

In [33]:
docs = retriever.invoke("How many casual leaves are employees entitled to?")

context = "\n\n".join(doc.page_content for doc in docs)

print(context)

contract employees, probationary employees, interns, and consultants on the company payroll. Employees
seconded from third-party vendors are governed by the terms of their respective contracts.
Leave Cycle: Financial Year (01 April to 31 March)

month, provided the employee has worked for a minimum of 20 days during the previous month.
 For employees joining or separating during the leave cycle, all leave will be calculated on a pro-rata basis.
 Casual Leave and Sick Leave cannot be carried forward to the next financial year, nor can they be
encashed. Any balance remaining at the end of the financial year (31 March) will lapse automatically.

15% of CTC
L7
Senior Manager
Rs. 60.0L to Rs. 80.0L
18% of CTC
L8
Director
Rs. 80.0L to Rs. 1.2Cr
20% of CTC
L9
Vice President
Rs. 1.2Cr to Rs. 2.0Cr
25% of CTC
L10
C-Suite
Rs. 2.0Cr and above
30% or more of CTC
STATUTORY BONUS
Employees whose gross monthly salary is at or below Rs. 21,000 (or such other amount as may be prescribed
under law fro

In [34]:
for chunk in chunks:
    if "Casual Leave (CL)" in chunk.page_content:
        print(chunk.page_content)
        print("="*100)

Zyro Dynamics Pvt. Ltd.
Leave Policy
Doc Code: ZDL-HR-002
Confidential — For Internal Use Only
Page 2
ANNUAL LEAVE ENTITLEMENT
 Leave Type
 Annual Entitlement
 Carry Forward
 Encashable
Earned Leave (EL)
15 days (after 1 year); 1.25
days/month
Up to 45 days
Yes
Casual Leave (CL)
8 days
Not allowed
Not applicable
Sick Leave (SL)
10 days
Not allowed
Not applicable
Maternity Leave
26 weeks
Not applicable
Not applicable
Paternity Leave
10 days
Not applicable
Not applicable
Bereavement Leave
5 days


In [35]:
docs = retriever.invoke("How many casual leaves are employees entitled to?")

for i, doc in enumerate(docs):
    if "Casual Leave (CL)" in doc.page_content:
        print("FOUND IT! 🎉")
        print(doc.page_content)

In [36]:
docs = retriever.invoke("casual leave")
print(len(docs))

20


In [37]:
docs = vectorstore.similarity_search("casual leave", k=3)

for doc in docs:
    print(doc.page_content[:500])
    print("="*50)

month, provided the employee has worked for a minimum of 20 days during the previous month.
 For employees joining or separating during the leave cycle, all leave will be calculated on a pro-rata basis.
 Casual Leave and Sick Leave cannot be carried forward to the next financial year, nor can they be
encashed. Any balance remaining at the end of the financial year (31 March) will lapse automatically.
 Employees are not permitted to avail more than 2 consecutive days of Casual Leave. If additional days are
required, they must be clubbed with Earned Leave.
 Sick Leave taken for more than 2 consecutive days requires a Medical Certificate from a registered medical
practitioner, to be submitted within 3 working days of returning to work.
 Leave cannot be availed during the notice period unless specifically approved by the reporting manager and
HR.
EARNED LEAVE (EL)
Eligibility and Accrual
Not applicable
Bereavement Leave
5 days
Not applicable
Not applicable
Marriage Leave
3 days (once 

In [38]:
docs = vectorstore.similarity_search(
    "8 days casual leave",
    k=10
)

for doc in docs:
    print(doc.page_content)
    print("="*50)

 Employees are not permitted to avail more than 2 consecutive days of Casual Leave. If additional days are
required, they must be clubbed with Earned Leave.
 Sick Leave taken for more than 2 consecutive days requires a Medical Certificate from a registered medical
practitioner, to be submitted within 3 working days of returning to work.
 Leave cannot be availed during the notice period unless specifically approved by the reporting manager and
HR.
EARNED LEAVE (EL)
Eligibility and Accrual
month, provided the employee has worked for a minimum of 20 days during the previous month.
 For employees joining or separating during the leave cycle, all leave will be calculated on a pro-rata basis.
 Casual Leave and Sick Leave cannot be carried forward to the next financial year, nor can they be
encashed. Any balance remaining at the end of the financial year (31 March) will lapse automatically.
Not applicable
Bereavement Leave
5 days
Not applicable
Not applicable
Marriage Leave
3 days (once 

In [39]:
context = """
Casual Leave (CL): 8 days
Sick Leave (SL): 10 days
Maternity Leave: 26 weeks
Paternity Leave: 10 days
"""

prompt = f"""
Answer from the context only.

Context:
{context}

Question:
How many casual leaves are employees entitled to?

Answer:
"""

response = llm.invoke(prompt)

print(response.content)

8 days


In [40]:
print(ask_hr_bot("Sick Leave (SL)"))

Sick Leave (SL) has an annual entitlement of 10 days. It cannot be carried forward to the next financial year, nor can it be encashed. Any balance remaining at the end of the financial year (31 March) will lapse automatically. Sick Leave taken for more than 2 consecutive days requires a Medical Certificate from a registered medical practitioner, to be submitted within 3 working days of returning to work.


In [41]:
print(ask_hr_bot("Maternity Leave"))

Female employees who have completed a minimum of 80 days of service in the 12 months preceding the expected date of delivery are entitled to 26 weeks of paid Maternity Leave, in accordance with the Maternity Benefit (Amendment) Act, 2017. This entitlement applies to the first two live births. For a third child, the entitlement is 12 weeks. Up to 8 weeks of pre-natal leave may be availed prior to the expected delivery date. Provisions for surrogacy and adoption are handled on a case-by-case basis.


In [42]:
print(ask_hr_bot("Paternity Leave"))

Male employees and same-sex partners are entitled to 10 consecutive days of paid Paternity Leave, to be availed within 90 days of the birth or legal adoption of a child. This is a one-time entitlement per child and cannot be split, subject to a minimum balance of 5 EL days being retained after encashment. The request must be raised through the ZyroHR portal.


In [43]:
question = "How many casual leaves are employees entitled to?"

docs = retriever.invoke(question)

print("Retrieved docs:", len(docs))

for i, doc in enumerate(docs):
    print(f"\n===== DOC {i+1} =====")
    print(doc.page_content[:500])

Retrieved docs: 20

===== DOC 1 =====
contract employees, probationary employees, interns, and consultants on the company payroll. Employees
seconded from third-party vendors are governed by the terms of their respective contracts.
Leave Cycle: Financial Year (01 April to 31 March)

===== DOC 2 =====
month, provided the employee has worked for a minimum of 20 days during the previous month.
 For employees joining or separating during the leave cycle, all leave will be calculated on a pro-rata basis.
 Casual Leave and Sick Leave cannot be carried forward to the next financial year, nor can they be
encashed. Any balance remaining at the end of the financial year (31 March) will lapse automatically.

===== DOC 3 =====
15% of CTC
L7
Senior Manager
Rs. 60.0L to Rs. 80.0L
18% of CTC
L8
Director
Rs. 80.0L to Rs. 1.2Cr
20% of CTC
L9
Vice President
Rs. 1.2Cr to Rs. 2.0Cr
25% of CTC
L10
C-Suite
Rs. 2.0Cr and above
30% or more of CTC
STATUTORY BONUS
Employees whose gross monthly salary is at or

In [44]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()

os.environ["LANGCHAIN_API_KEY"] = user_secrets.get_secret("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "zyro-rag-challenge"

print("LangSmith connected ✅")

LangSmith connected ✅


In [51]:
print(ask_hr_bot("What is the maternity leave policy?"))

Female employees who have completed a minimum of 80 days of service in the 12 months preceding the expected date of delivery are entitled to 26 weeks of paid Maternity Leave. This entitlement applies to the first two live births. For a third child, the entitlement is 12 weeks. Up to 8 weeks of pre-natal leave may be availed prior to the expected delivery date. Provisions for surrogacy and adoption are handled on a case-by-case basis.


In [46]:
import os

print("LANGCHAIN_API_KEY:", "SET" if os.getenv("LANGCHAIN_API_KEY") else "NOT SET")
print("LANGCHAIN_TRACING_V2:", os.getenv("LANGCHAIN_TRACING_V2"))
print("LANGCHAIN_PROJECT:", os.getenv("LANGCHAIN_PROJECT"))

LANGCHAIN_API_KEY: SET
LANGCHAIN_TRACING_V2: true
LANGCHAIN_PROJECT: zyro-rag-challenge


In [48]:
llm

ChatGroq(metadata={'versions': {'langchain-core': '1.4.6', 'langchain': '1.2.15'}}, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7cca99e19f10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7cca9a009580>, model_name='llama-3.3-70b-versatile', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [50]:
ask_hr_bot

<function __main__.ask_hr_bot(question)>